# 1. Project Introduction

Welcome! In this notebook, we will explore **XGBoost** (Extreme Gradient Boosting), one of the most popular and powerful algorithms for tabular data.

### What is XGBoost?
* It is a **supervised learning** classifier.
* Like AdaBoost, it uses boosting. However, instead of adjusting sample weights, it fits new trees to the **residuals** (the errors/gradients) of the previous trees. This is called **Gradient Boosting**.
* **Extreme**: It is called "Extreme" because it is designed to be highly optimized, fast, and handles missing values and regularization to prevent overfitting automatically.

### Why does it exist?
* It was designed to push the limits of computing speed and model performance, and is a dominant algorithm in machine learning competitions.

### Real-World Use Cases:
* **Risk Scoring**: Predicting default risks.
* **Search Ranking**: Sorting search engine results based on relevance.


# 2. Problem Statement

* **Goal**: Predict whether a bank customer will default on a personal loan (**1**) or repay (**0**).
* **Business Value**: Protects financial institutions from toxic credit defaults.


In [ ]:
# Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn import metrics


# 4. Create Synthetic Dataset

We define features for **100 applications**.
* **Age**: Applicant age.
* **Annual_Income_K**: Income in k$.
* **Credit_Score**: Credit score.
* **Debt_Ratio**: Total debt divided by annual income.
* **Loan_Default**: Label.


In [ ]:
# Hardcoded borrower loan dataset
age = [
    25, 45, 30, 55, 22, 40, 60, 27, 33, 48, 26, 35, 52, 29, 42, 50, 31, 38, 44, 23,
    26, 46, 31, 56, 23, 41, 61, 28, 34, 49, 27, 36, 53, 30, 43, 51, 32, 39, 45, 24,
    24, 44, 29, 54, 21, 39, 59, 26, 32, 47, 25, 34, 51, 28, 41, 49, 30, 37, 43, 22,
    25, 45, 30, 55, 22, 40, 60, 27, 33, 48, 26, 35, 52, 29, 42, 50, 31, 38, 44, 23,
    30, 35, 40, 45, 50, 25, 28, 32, 38, 42, 48, 52, 55, 60, 22, 24, 29, 31, 34, 37
]

income = [
    30, 85, 42, 120, 25, 60, 150, 38, 55, 90, 40, 75, 110, 50, 80, 100, 65, 70, 95, 32,
    32, 88, 44, 122, 27, 62, 152, 40, 57, 92, 42, 77, 112, 52, 82, 102, 67, 72, 97, 34,
    28, 83, 40, 118, 23, 58, 148, 36, 53, 88, 38, 73, 108, 48, 78, 98,  63, 68, 93, 30,
    30, 85, 42, 120, 25, 60, 150, 38, 55, 90, 40, 75, 110, 50, 80, 100, 65, 70, 95, 32,
    40, 50, 60, 70,  80,  35, 45, 55, 65, 75, 85,  95, 105, 120, 28, 30, 42, 48, 52, 58
]

credit_score = [
    550, 710, 620, 780, 500, 650, 790, 580, 600, 680, 590, 640, 720, 610, 660, 700, 630, 650, 670, 530,
    555, 715, 625, 785, 505, 655, 795, 585, 605, 685, 595, 645, 725, 615, 665, 705, 635, 655, 675, 535,
    545, 705, 615, 775, 495, 645, 785, 575, 595, 675, 585, 635, 715, 605, 655, 695, 625, 645, 665, 525,
    550, 710, 620, 780, 500, 650, 790, 580, 600, 680, 590, 640, 720, 610, 660, 700, 630, 650, 670, 530,
    520, 630, 700, 710, 790, 550, 580, 610, 640, 680, 710, 720, 750, 760, 500, 530, 590, 620, 640, 660
]

debt_ratio = [
    0.45, 0.12, 0.35, 0.08, 0.60, 0.28, 0.05, 0.40, 0.32, 0.22, 0.38, 0.25, 0.15, 0.30, 0.24, 0.18, 0.29, 0.27, 0.20, 0.50,
    0.44, 0.11, 0.34, 0.07, 0.58, 0.27, 0.04, 0.39, 0.31, 0.21, 0.37, 0.24, 0.14, 0.29, 0.23, 0.17, 0.28, 0.26, 0.19, 0.49,
    0.46, 0.13, 0.36, 0.09, 0.62, 0.29, 0.06, 0.41, 0.33, 0.23, 0.39, 0.26, 0.16, 0.31, 0.25, 0.19, 0.30, 0.28, 0.21, 0.51,
    0.45, 0.12, 0.35, 0.08, 0.60, 0.28, 0.05, 0.40, 0.32, 0.22, 0.38, 0.25, 0.15, 0.30, 0.24, 0.18, 0.29, 0.27, 0.20, 0.50,
    0.55, 0.42, 0.30, 0.25, 0.15, 0.52, 0.48, 0.38, 0.33, 0.27, 0.22, 0.18, 0.12, 0.08, 0.61, 0.58, 0.44, 0.36, 0.32, 0.29
]

loan_default = [
    1, 0, 0, 0, 1, 0, 0, 1, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1,
    1, 0, 0, 0, 1, 0, 0, 1, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1,
    1, 0, 0, 0, 1, 0, 0, 1, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1,
    1, 0, 0, 0, 1, 0, 0, 1, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1,
    1, 1, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0
]

df = pd.DataFrame({
    'Age': age,
    'Annual_Income_K': income,
    'Credit_Score': credit_score,
    'Debt_Ratio': debt_ratio,
    'Loan_Default': loan_default
})

print("Shape:", df.shape)
print(df.head())


# 5. Exploratory Data Analysis (EDA)


In [ ]:
# Chart 1: Credit Score vs Debt Ratio colored by Loan Default
plt.figure(figsize=(8, 5))
sns.scatterplot(x='Credit_Score', y='Debt_Ratio', hue='Loan_Default', data=df, palette='coolwarm', s=80)
plt.title('Credit Score vs. Debt Ratio')
plt.xlabel('Credit Score')
plt.ylabel('Debt Ratio')
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()


### What Did We Observe?
* Defaulters cluster at low credit scores and high debt ratios.


In [ ]:
# Cleaning check
print("Null count:", df.isnull().sum().sum())


In [ ]:
# Feature Selection
X = df[['Age', 'Annual_Income_K', 'Credit_Score', 'Debt_Ratio']]
y = df['Loan_Default']


In [ ]:
# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


# 9. Model Building

* **How it works**: XGBoost trains a sequence of trees. Instead of calculating sample weights, it computes the **residuals (errors)** of the current predictions and trains a new tree to predict those residuals. The learning rate (or `eta`) weights the contributions of each tree.


In [ ]:
# Initialize XGBoost Classifier with 30 estimators and learning rate = 0.1
model = XGBClassifier(n_estimators=30, max_depth=3, learning_rate=0.1, random_state=42)


In [ ]:
# Train XGBoost model
model.fit(X_train, y_train)


In [ ]:
# Predict labels
predictions = model.predict(X_test)


In [ ]:
# Compute metrics
accuracy = metrics.accuracy_score(y_test, predictions)
precision = metrics.precision_score(y_test, predictions)
recall = metrics.recall_score(y_test, predictions)
f1 = metrics.f1_score(y_test, predictions)
conf_matrix = metrics.confusion_matrix(y_test, predictions)

# Print metrics in plain English
print(f"Accuracy Score: {accuracy:.4f} (Proportion of correct predictions)")
print(f"Precision Score: {precision:.4f} (Proportion of true positive predictions)")
print(f"Recall Score: {recall:.4f} (Proportion of actual positives caught)")
print(f"F1 Score: {f1:.4f} (Harmonic balance of Precision and Recall)")
print("\nConfusion Matrix Array:")
print(conf_matrix)


# 13. Visualizing Model Performance

We will plot:
1. **Confusion Matrix Heatmap**.
2. **Feature Importance Plot**: Contributions based on splitting gain.


In [ ]:
# Plot 1: Confusion Matrix Heatmap
conf_matrix = metrics.confusion_matrix(y_test, predictions)
plt.figure(figsize=(6, 4))
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Purples', 
            xticklabels=['Predicted Repay', 'Predicted Default'], 
            yticklabels=['Actual Repay', 'Actual Default'])
plt.title('XGBoost Confusion Matrix')
plt.show()


In [ ]:
# Plot 2: Feature Importances
plt.figure(figsize=(6, 4))
sns.barplot(x=model.feature_importances_, y=X.columns, palette='viridis')
plt.title('XGBoost Feature Importances')
plt.xlabel('Importance score')
plt.ylabel('Feature')
plt.grid(axis='x', linestyle='--', alpha=0.6)
plt.show()


### What Did We Observe?
* `Credit_Score` displays the highest split contribution score.


# 14. Model Interpretation

* **Gradient Boosting**: Each tree corrects residuals of previous trees.
* **Regularization**: XGBoost applies L1/L2 penalties internally to reduce overfitting.


# 15. Conclusion
* XGBoost handles complex non-linear tabular datasets exceptionally well.


# 16. Beginner ML Dictionary

Here are simple, one-sentence explanations of common Machine Learning terms to help you review:

* **Feature**: An input variable or column in your dataset used to make predictions (e.g., hours studied).
* **Target**: The output variable or label you want the model to predict (e.g., final exam score).
* **Training Data**: The portion of the dataset used to teach the model and find patterns.
* **Testing Data**: The portion of the dataset held back to evaluate how well the model performs on new, unseen data.
* **Prediction**: The output value generated by the trained model when given new input features.
* **Overfitting**: A scenario where the model learns the training data too well, including its noise, and performs poorly on new data.
* **Underfitting**: A scenario where the model is too simple to learn the underlying patterns in the training data, leading to poor performance on both training and test data.
* **Model**: The mathematical representation of the patterns learned from the training data by the algorithm.
* **Algorithm**: The set of rules or mathematical procedures followed to build the model from the data (e.g., Linear Regression).
* **Accuracy**: The percentage of correct predictions made by a classification model.
* **Cluster**: A group of similar data points grouped together by an unsupervised learning algorithm based on their characteristics.
* **Centroid**: The center point of a cluster, representing the average location of all data points belonging to that cluster.
